In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
path = '/workspaces/crmprtd/update_sheet/'
df = pd.read_excel(path + '20250918-Metadata-AllRequiredChanges.xlsx', header = 1)   # pandas automatically uses openpyxl
df_fern = df[df["Network Name"] == '-> BC-FERN']
df_pcds = df_fern[['Station ID', 'Unique Names','Unique Location (String)', 'Native ID']].reset_index(drop=True)
df_ted = df_fern[['Station_name','native_id', 'lat', 'lon', 'Elevation']].reset_index(drop=True)


df_pcds['Station_name'] = df_fern['Unique Names'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df_pcds = df_pcds.drop(columns=['Unique Names'])

df_pcds['native_id'] = df_fern['Native ID'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df_pcds = df_pcds.drop(columns=['Native ID'])

In [4]:
import re

def parse_lat_lon_elev(text):
    # regex to capture: lon value, W/E, lat value, N/S, elevation
    pattern = r'([\d\.]+)\s*([WE]),\s*([\d\.]+)\s*([NS]),\s*Elev\.\s*([\d\.]+)'
    m = re.search(pattern, text)
    
    if not m:
        return None, None, None

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    # Convert numbers
    lon = float(lon_val) * (-1 if lon_dir == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir == "S" else 1)
    elev = float(elev_val)

    return lat, lon, elev

# Apply to dataframe
df_pcds[['lat', 'lon', 'Elevation']] = df_pcds['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_lat_lon_elev(x))
)
df_pcds = df_pcds.drop(columns=['Unique Location (String)'])

In [5]:
# 1. Rename df_pcds columns to match df_ted
df_pcds_renamed = df_pcds.rename(columns={
    'pcds_sation_name': 'Station_name',
    'pcds_native_id': 'native_id',
    'pcds_lat': 'lat',
    'pcds_lon': 'lon',
    'pcds_elev': 'Elevation'
})

# 2. Merge on Station_name
merged = df_pcds_renamed.merge(
    df_ted,
    on='Station_name',
    how='inner',
    suffixes=('_pcds', '_ted')
)

# 3. Compare key columns
merged['lat_match']  = merged['lat_pcds']  == merged['lat_ted']
merged['lon_match']  = merged['lon_pcds']  == merged['lon_ted']
merged['elev_match'] = merged['Elevation_pcds'] == merged['Elevation_ted']
merged['id_match']   = merged['native_id_pcds'] == merged['native_id_ted']

# 4. Whether all match
merged['all_match'] = (
    merged['lat_match'] &
    merged['lon_match'] &
    merged['elev_match'] &
    merged['id_match']
)

# 5. Show results
merged[['Station_name', 'lat_match', 'lon_match', 'elev_match', 'id_match', 'all_match']]



,Station_name,lat_match,lon_match,elev_match,id_match,all_match
0,Atlin School,True,True,True,True,True
1,Cassiar,True,True,True,True,True
2,Monarch-1,True,True,True,True,True
3,Northern Dancer,True,True,True,True,True
4,Goose Lake Rd,True,True,True,True,True
...,...,...,...,...,...,...
59,Mayson Lake - M2,True,True,True,True,True
60,Mayson Lake - M3,True,True,True,True,True
61,Mayson Lake - M4,False,False,False,True,False
62,Mayson Lake - M5,True,True,True,True,True


In [10]:
# Filter rows where not all match
mismatched = merged[merged['all_match'] == False]

# Show relevant columns
mismatched[['Station_name', 
            'lat_pcds', 'lat_ted', 
            'lon_pcds', 'lon_ted', 
            'Elevation_pcds', 'Elevation_ted', 
            'native_id_pcds', 'native_id_ted']]

df_ted.loc[df_ted['Station_name'] == 'Mayson Lake - M4', 'lon'] = '-122.058'

df_ted['Elevation'] = pd.to_numeric(df_ted['Elevation'], errors='coerce')

save_path = './comparison_forms/'
df_ted.to_csv(save_path + '0.1_64_fern_station_insert_info.csv', index=False)
df_ted


,Station_name,native_id,lat,lon,Elevation
0,Atlin School,AtlSch,59.574,-133.687,705.0
1,Cassiar,CasWx,59.251,-129.86,1577.0
2,Monarch-1,Mon1,59.544,-133.617,1410.0
3,Northern Dancer,NDWx,60.008,-131.6,1628.0
4,Goose Lake Rd,Goose,50.5626,-120.43,1147.0
...,...,...,...,...,...
59,Mayson Lake - M2,M2,51.216528,-120.388667,1277.0
60,Mayson Lake - M3,M3,51.217611,-120.401694,1290.0
61,Mayson Lake - M4,M4,51.219111,-122.058,NaN
62,Mayson Lake - M5,M5,51.212361,-120.412222,1314.0


For the station `SeebachWx`, the lon should be `-122.058` in Ted's sheet, the left pcds part is correct

`Mayson Lake - M4` has no elev information

### Insert into db

In [15]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [18]:
from sqlalchemy import func

stations_created = []

# for _, row in df_ted.iloc[0:2].iterrows():
for _, row in df_ted.iterrows():
    
    name = row['Station_name']
    nid  = row['native_id']

    # 1. Create Station
    st = Station(
        native_id=nid,
        publish=True,
        network_id=11)


    session.add(st)
    session.flush()  # 得到 st.id

    h = History(
        station_id=st.id,
        station_name=name,
        lat=row['lat'],
        lon=row['lon'],
        elevation=row['Elevation'],
        province="BC",
        country="CA",
        the_geom=func.ST_SetSRID(func.ST_MakePoint(row['lon'], row['lat']), 4326))

    session.add(h)

    stations_created.append((name, st.id))

session.commit()

print("Created:", stations_created)

Created: [('Atlin School', 14779), ('Cassiar', 14780), ('Monarch-1', 14781), ('Northern Dancer', 14782), ('Goose Lake Rd', 14783), ('Isobel Lake (IDFxh2)', 14784), ('LDB - Lower grassland', 14785), ('LDB - LowerUpper grassland', 14786), ('LDB - Middle grassland', 14787), ('LDB - Upper grassland', 14788), ('LDB - UpperMiddle grassland', 14789), ('Arrowstone', 14790), ('Chase Creek', 14791), ('Sicamouse Creek - WV1', 14792), ('Upper Penticton - PK', 14793), ('Upper Penticton Creek - P1', 14794), ('Upper Penticton Creek - P5', 14795), ('Upper Penticton Creek - PB', 14796), ('Upper Penticton Creek - PC', 14797), ('Upper Penticton Creek - PJ', 14798), ('Upper Penticton Creek - PL', 14799), ('CrookedLk', 14800), ('DavisPGTIS', 14801), ('Hudson Bay Stn', 14802), ('IBB1Wx', 14803), ('IBB2Wx', 14804), ('IBB3Wx', 14805), ('Inklin', 14806), ('KennedySidingCC', 14807), ('KennedySidingFor', 14808), ('SeebachWx', 14809), ('SumWxCC', 14810), ('Tamarac', 14811), ('Ape Lake', 14812), ('Cain Ridge_Run',

In [22]:
# Begin a SAVEPOINT (mini-transaction)
session.begin_nested()

update_sql = sa.text("""
    UPDATE meta_history
    SET the_geom = ST_SetSRID(ST_MakePoint(lon, lat), 4326)
    WHERE station_name = :name AND the_geom IS NULL
""")

# Run the updates
for _, row in df_ted.iterrows():
    session.execute(update_sql, {"name": row["Station_name"]})

# Query updated rows to inspect
preview = session.execute(sa.text("""
    SELECT station_name, lat, lon, ST_AsText(the_geom)
    FROM meta_history
    WHERE the_geom IS NOT NULL
      AND station_name IN :names
"""), {"names": tuple(df_ted['Station_name'])}).fetchall()

for p in preview:
    print(p)

# Rollback the SAVEPOINT — discards all updates
session.rollback()

('CLAYTON FALLS', Decimal('52.279579'), Decimal('-126.890543'), 'POINT(-126.890543 52.279579)')
('Cassiar', Decimal('59.24722'), Decimal('-129.6575'), 'POINT(-129.6575 59.24722)')
('Atlin School', Decimal('59.574'), Decimal('-133.687'), 'POINT(-133.687 59.574)')
('Cassiar', Decimal('59.251'), Decimal('-129.86'), 'POINT(-129.86 59.251)')
('Monarch-1', Decimal('59.544'), Decimal('-133.617'), 'POINT(-133.617 59.544)')
('Northern Dancer', Decimal('60.008'), Decimal('-131.6'), 'POINT(-131.6 60.008)')
('Goose Lake Rd', Decimal('50.5626'), Decimal('-120.43'), 'POINT(-120.43 50.5626)')
('Isobel Lake (IDFxh2)', Decimal('50.8406'), Decimal('-120.4231'), 'POINT(-120.4231 50.8406)')
('LDB - Lower grassland', Decimal('50.7285'), Decimal('-120.4383'), 'POINT(-120.4383 50.7285)')
('LDB - LowerUpper grassland', Decimal('50.7868'), Decimal('-120.4483'), 'POINT(-120.4483 50.7868)')
('LDB - Middle grassland', Decimal('50.7533'), Decimal('-120.4158'), 'POINT(-120.4158 50.7533)')
('LDB - Upper grassland', 

In [23]:

for _, row in df_ted.iterrows():
    session.execute(update_sql, {"name": row["Station_name"]})

session.commit()

In [24]:
update_sql = sa.text("""
    UPDATE meta_history
    SET the_geom = ST_SetSRID(ST_MakePoint(lon, lat), 4326)
    WHERE station_name = :name AND the_geom IS NULL
""")

session.execute(update_sql, {"name": "Dennis"})
session.commit()

In [27]:
from crmprtd import Row
help(Row)

Help on class Row in module crmprtd:

class Row(builtins.tuple)
 |  Row(time, val, variable_name, unit, network_name, station_id, lat, lon)
 |
 |  Row(time, val, variable_name, unit, network_name, station_id, lat, lon)
 |
 |  Method resolution order:
 |      Row
 |      builtins.tuple
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __getnewargs__(self) from collections.Row
 |      Return self as a plain tuple.  Used by copy and pickle.
 |
 |  __replace__ = _replace(self, /, **kwds)
 |
 |  __repr__(self) from collections.Row
 |      Return a nicely formatted representation string
 |
 |  _asdict(self) from collections.Row
 |      Return a new dict which maps field names to their values.
 |
 |  _replace(self, /, **kwds) from collections.Row
 |      Return a new Row object replacing specified fields with new values
 |
 |  ----------------------------------------------------------------------
 |  Class methods defined here:
 |
 |  _make(iterable) from collections.Row
 |      Mak

In [106]:
from pycds import Station
# help(Station)
Station.__table__.columns.keys()


['station_id',
 'native_id',
 'network_id',
 'min_obs_time',
 'max_obs_time',
 'publish',
 'mod_time',
 'mod_user']

In [26]:
from pycds import Station, History
help(History)
# History.__table__.columns.keys()


Help on class History in module pycds.orm.tables:

class History(sqlalchemy.orm.decl_api.Base)
 |  History(**kwargs)
 |
 |  This class maps to the table which represents a history record for
 |  a weather station. Since a station can potentially (and do) move
 |  small distances (e.g. from one end of the airport runway to
 |  another) or change the frequency of its observations, this table
 |  records the details of those changes.
 |
 |  WARNING: The GeoAlchemy2 `Geometry` column (attribute `the_geom`) forces
 |  all reads on that column to be wrapped with Postgis function `ST_AsEWKB`.
 |  This may or may not be desirable for all use cases, specifically views.
 |  See the GeoAlchemy2 documentation for details.
 |
 |  Method resolution order:
 |      History
 |      sqlalchemy.orm.decl_api.Base
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, **kwargs) from sqlalchemy.orm.instrumentation
 |      A simple constructor that allows initialization from kwargs.
 |
 |

In [25]:
from pycds import Variable
Variable.__table__.columns.keys()


['vars_id',
 'net_var_name',
 'unit',
 'standard_name',
 'cell_method',
 'precision',
 'long_description',
 'display_name',
 'short_name',
 'network_id',
 'mod_time',
 'mod_user']

In [99]:
from crmprtd import Row
help(Row)

Help on class Row in module crmprtd:

class Row(builtins.tuple)
 |  Row(time, val, variable_name, unit, network_name, station_id, lat, lon)
 |
 |  Row(time, val, variable_name, unit, network_name, station_id, lat, lon)
 |
 |  Method resolution order:
 |      Row
 |      builtins.tuple
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __getnewargs__(self) from collections.Row
 |      Return self as a plain tuple.  Used by copy and pickle.
 |
 |  __replace__ = _replace(self, /, **kwds)
 |
 |  __repr__(self) from collections.Row
 |      Return a nicely formatted representation string
 |
 |  _asdict(self) from collections.Row
 |      Return a new dict which maps field names to their values.
 |
 |  _replace(self, /, **kwds) from collections.Row
 |      Return a new Row object replacing specified fields with new values
 |
 |  ----------------------------------------------------------------------
 |  Class methods defined here:
 |
 |  _make(iterable) from collections.Row
 |      Mak